# AnyTimeVisionenc — Train MSDNet on vision datasets

Reference: [MSDNet-PyTorch](https://github.com/kalviny/MSDNet-PyTorch)

Flow: **Setup → HF login → Prepare data → Train (push HF) → Verify**

## 1. Setup

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_PATH = "/content/spd"
    !git clone https://github.com/wicaksonolxn/spd.git $REPO_PATH 2>/dev/null || (cd $REPO_PATH && git pull)
except Exception:
    REPO_PATH = ".."
%cd $REPO_PATH

In [ ]:
!pip install -q -r AnyTimeVisionenc/requirements.txt

## 2. HF login

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HF_USER"] = userdata.get("HF_USER", "wicaksonolxn")
except Exception:
    pass
from huggingface_hub import login, whoami

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Logged in as:", whoami().get("name", "?"))

## 3. Prepare data

In [ ]:
import sys

sys.path.insert(0, ".")
from AnyTimeVisionenc import prepare_cifar10, prepare_cifar100

prepare_cifar10()
prepare_cifar100()
# ImageNet manual — see prepare_imagenet() output

## 4. Train

In [ ]:
from AnyTimeVisionenc import train

# Smallest first (CIFAR-10, ~1-2hr on T4):
ckpt = train("cifar10", epochs=300)

# All:
# train_all()

# Custom:
# train('cifar100', epochs=200, batch_size=128, lr=0.1, n_blocks=7)

## 5. Verify HF push

In [ ]:
from huggingface_hub import list_repo_files
from AnyTimeVisionenc.config import hf_repo_for, DATASETS

for d in DATASETS:
    repo = hf_repo_for(d)
    try:
        files = list_repo_files(repo)
        print(f"  {repo}: {len(files)} files")
    except Exception as e:
        print(f"  {repo}: not pushed yet ({e})")